# Task 1 — Data Cleaning & Preprocessing

**Objectives**
- Import a dataset using Python and inspect its structure.
- Identify missing values, duplicate records, and inconsistent data entries.
- Clean the dataset by handling null values, removing duplicates, and correcting data types.
- Prepare the data for further analysis using Pandas.
- **Bonus:** Save the cleaned dataset as a new CSV file.

**Dataset:** `data/online_retail_orders_raw.csv` — a synthetic online-retail orders
export (220 orders + 12 duplicated rows = 232 rows) with realistic messiness:
missing values, duplicate rows, inconsistent text casing/whitespace, prices
stored as text with currency symbols, three different date formats, and a
handful of invalid (negative / text) quantities.


In [177]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## Step 1 — Import the dataset and inspect its structure

In [178]:
df = pd.read_csv("data/online_retail_orders_raw.csv")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(8)

Shape: 232 rows x 10 columns


,OrderID,CustomerName,Email,Product,Category,Quantity,Price,OrderDate,ShippingCity,PaymentMethod
0,1155,Mia Wilson,mia.wilson@example.com,Bluetooth Speaker,Electronics,3,46.1,"Feb 15, 2024",new york,Debit Card
1,1091,AMIR CHEN,amir.chen@example.com,Bluetooth Speaker,Electronics,1 units,44.1,"Jul 01, 2024",Chicago,Credit Card
2,1215,Wei Silva,wei.silva@example.com,Bluetooth Speaker,Electronics,2,$47.02,04/08/2024,New York,credit card
3,1025,Liam Garcia,liam.garcia@example.com,Notebook Set,Stationery,4,NaN,"Jul 03, 2024",Miami,Credit Card
4,1216,diego rossi,diego.rossi@example.com,Ballpoint Pens (12pk),Stationery,2,6.22,2024-10-17,Seattle,credit card
5,1067,Amir Silva,amir.silva@example.com,Sunglasses,Accessories,1,30.59,2024-05-13,MIAMI,Cash
6,1004,Amir Brown,amir.brown@example.com,Ballpoint Pens (12pk),Stationery,2,$6.54,"Aug 21, 2024",CHICAGO,paypal
7,1012,Mia Nguyen,mia.nguyen@example.com,Wireless Mouse,Electronics,2,19.11,29/03/2024,Seattle,cash


In [179]:
df.dtypes

OrderID          int64
CustomerName       str
Email              str
Product            str
Category           str
Quantity           str
Price              str
OrderDate          str
ShippingCity       str
PaymentMethod      str
dtype: object

In [180]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 232 entries, 0 to 231
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   OrderID        232 non-null    int64
 1   CustomerName   226 non-null    str  
 2   Email          221 non-null    str  
 3   Product        232 non-null    str  
 4   Category       232 non-null    str  
 5   Quantity       224 non-null    str  
 6   Price          222 non-null    str  
 7   OrderDate      232 non-null    str  
 8   ShippingCity   220 non-null    str  
 9   PaymentMethod  226 non-null    str  
dtypes: int64(1), str(9)
memory usage: 18.3 KB


In [181]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
OrderID,232.0,NaN,NaN,NaN,1109.310345,63.307798,1001.0,1055.75,1106.5,1164.25,1220.0
CustomerName,226,181,Lucas Brown,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Email,221,140,fatima.garcia@example.com,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product,232,10,Sunglasses,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,232,5,Home,51,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,224,12,3,48,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price,222,209,$20.91,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OrderDate,232,196,"Apr 29, 2024",4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ShippingCity,220,17,seattle,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PaymentMethod,226,9,Cash,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 2 — Identify missing values, duplicates, and inconsistent entries

In [182]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

,missing_count,missing_pct
OrderID,0,0.0
CustomerName,6,2.6
Email,11,4.7
Product,0,0.0
Category,0,0.0
Quantity,8,3.4
Price,10,4.3
OrderDate,0,0.0
ShippingCity,12,5.2
PaymentMethod,6,2.6


In [183]:
n_dupes = df.duplicated().sum()
print(f"Fully duplicated rows: {n_dupes}")

Fully duplicated rows: 12


In [184]:
for col in ["Category", "PaymentMethod", "ShippingCity"]:
    print(f"\n--- Unique raw values in '{col}' ---")
    print(sorted(df[col].dropna().unique()))


--- Unique raw values in 'Category' ---
['Accessories', 'Electronics', 'Fitness', 'Home', 'Stationery']

--- Unique raw values in 'PaymentMethod' ---
['CREDIT CARD', 'Cash', 'Credit Card', 'Debit Card', 'PayPal', 'cash ', 'credit card', 'debit card', 'paypal']

--- Unique raw values in 'ShippingCity' ---
[' New York ', ' miami', 'CHICAGO', 'Chicago', 'Houston', 'LOS ANGELES', 'Los Angeles', 'MIAMI', 'Miami', 'NEW YORK', 'New York', 'Seattle', 'chicago ', 'houston', 'los angeles', 'new york', 'seattle']


In [185]:
print("Sample OrderDate formats present:")
print(df["OrderDate"].sample(6, random_state=1).tolist())

Sample OrderDate formats present:
['2024-06-27', '14/02/2024', '20/01/2024', '28/05/2024', 'Aug 03, 2024', '31/03/2024']


In [186]:
print("Non-numeric-looking Quantity values:")
print(df.loc[pd.to_numeric(df["Quantity"], errors="coerce").isna() & df["Quantity"].notna(), "Quantity"].unique())

print("\nNegative quantities (data entry errors):")
qty_numeric = pd.to_numeric(df["Quantity"].astype(str).str.extract(r"(-?\d+)")[0], errors="coerce")
print((qty_numeric < 0).sum(), "rows with a negative quantity")

Non-numeric-looking Quantity values:
<StringArray>
['1 units', '4 units', '5 units', '3 units']
Length: 4, dtype: str

Negative quantities (data entry errors):
4 rows with a negative quantity


In [187]:
print("Sample Price values showing inconsistent formatting:")
print(df["Price"].astype(str).sample(8, random_state=3).tolist())

Sample Price values showing inconsistent formatting:
['$6.41', '38.05', ' 40.21 ', ' 38.13 ', nan, ' 38.33 ', ' 19.86 ', '21.46']


## Step 3 — Clean the dataset

**Strategy for each issue found above:**

| Issue | Column(s) | Action |
|---|---|---|
| Stray whitespace / mixed case | `CustomerName`, `ShippingCity`, `PaymentMethod`, `Category` | `.str.strip()` + consistent casing (title case for names/cities, title case for category/payment) |
| Price stored as text (`$45.50`, `" 6.22 "`) | `Price` | strip `$`/whitespace, cast to `float` |
| Quantity stored as text (`"3 units"`) or negative | `Quantity` | extract digits, take absolute value, cast to `int` |
| Mixed date formats | `OrderDate` | parse all 3 formats into one `datetime64` column |
| Missing `CustomerName` / `Email` / `ShippingCity` / `PaymentMethod` | those columns | fill with `"Unknown"` (categorical — safe to flag rather than guess) |
| Missing `Quantity` / `Price` | those columns | fill with the column **median** (numeric — median is robust to the outliers we saw) |
| Fully duplicated rows | whole row | `drop_duplicates()` |


In [188]:
df_clean = df.copy()

# 1) Strip whitespace and standardize casing on text columns
text_cols = ["CustomerName", "ShippingCity", "PaymentMethod", "Category", "Product"]
for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()
    df_clean.loc[df_clean[col].isin(["nan", "None", ""]), col] = np.nan

df_clean["CustomerName"] = df_clean["CustomerName"].str.title()
df_clean["ShippingCity"] = df_clean["ShippingCity"].str.title()
df_clean["PaymentMethod"] = df_clean["PaymentMethod"].str.title()
df_clean["Category"] = df_clean["Category"].str.title()

print("Cities after standardizing:", sorted(df_clean["ShippingCity"].dropna().unique()))
print("Payment methods after standardizing:", sorted(df_clean["PaymentMethod"].dropna().unique()))

Cities after standardizing: ['Chicago', 'Houston', 'Los Angeles', 'Miami', 'New York', 'Seattle']
Payment methods after standardizing: ['Cash', 'Credit Card', 'Debit Card', 'Paypal']


In [189]:
# 2) Fix Price: strip "$" and whitespace, convert to float
df_clean["Price"] = (
    df_clean["Price"].astype(str)
    .str.replace("$", "", regex=False)
    .str.strip()
    .replace({"nan": np.nan})
    .astype(float)
)
df_clean["Price"].describe()

count    222.000000
mean      31.587928
std       17.952166
min        6.000000
25%       19.000000
50%       29.530000
75%       44.265000
max       67.660000
Name: Price, dtype: float64

In [190]:
# 3) Fix Quantity: pull out the numeric part, force positive, convert to int (nullable)
df_clean["Quantity"] = (
    df_clean["Quantity"].astype(str).str.extract(r"(-?\d+)")[0].astype(float).abs()
)
df_clean["Quantity"] = df_clean["Quantity"].astype("Int64")  # nullable int, NaNs kept for now
df_clean["Quantity"].describe()

count       224.0
mean     2.933036
std      1.420534
min           1.0
25%           2.0
50%           3.0
75%           4.0
max           5.0
Name: Quantity, dtype: Float64

In [191]:
# 4) Parse the three different OrderDate formats into one datetime column
def parse_mixed_date(val):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%b %d, %Y"):
        try:
            return pd.to_datetime(val, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df_clean["OrderDate"] = df_clean["OrderDate"].apply(parse_mixed_date)
print("Unparsed dates:", df_clean["OrderDate"].isna().sum())
df_clean["OrderDate"].head()

Unparsed dates: 0


0   2024-02-15
1   2024-07-01
2   2024-08-04
3   2024-07-03
4   2024-10-17
Name: OrderDate, dtype: datetime64[us]

In [192]:
# 5) Handle missing values
# Categorical / identifier columns -> explicit "Unknown" flag (don't invent a name/city)
for col in ["CustomerName", "Email", "ShippingCity", "PaymentMethod"]:
    df_clean[col] = df_clean[col].fillna("Unknown")

# Numeric columns -> median imputation (robust to skew)
df_clean["Quantity"] = df_clean["Quantity"].fillna(df_clean["Quantity"].median()).astype(int)
df_clean["Price"] = df_clean["Price"].fillna(df_clean["Price"].median()).round(2)

print("Remaining missing values per column:")
print(df_clean.isnull().sum())

Remaining missing values per column:
OrderID          0
CustomerName     0
Email            0
Product          0
Category         0
Quantity         0
Price            0
OrderDate        0
ShippingCity     0
PaymentMethod    0
dtype: int64


In [193]:
# 6) Remove duplicate rows (based on OrderID, the true unique key, and full-row dupes)
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.drop_duplicates(subset="OrderID", keep="first")
after = len(df_clean)
print(f"Removed {before - after} duplicate rows  ({before} -> {after})")

Removed 12 duplicate rows  (232 -> 220)


In [194]:
# 7) Correct final data types
df_clean["OrderID"] = df_clean["OrderID"].astype(int)
df_clean["Quantity"] = df_clean["Quantity"].astype(int)
df_clean["Price"] = df_clean["Price"].astype(float)
df_clean["OrderDate"] = pd.to_datetime(df_clean["OrderDate"])

df_clean.dtypes

OrderID                   int64
CustomerName                str
Email                       str
Product                     str
Category                    str
Quantity                  int64
Price                   float64
OrderDate        datetime64[us]
ShippingCity                str
PaymentMethod               str
dtype: object

## Step 4 — Prepare the data for further analysis

In [195]:
df_clean = df_clean.sort_values("OrderDate").reset_index(drop=True)
df_clean["TotalAmount"] = (df_clean["Quantity"] * df_clean["Price"]).round(2)

df_clean.head(10)

,OrderID,CustomerName,Email,Product,Category,Quantity,Price,OrderDate,ShippingCity,PaymentMethod,TotalAmount
0,1188,Ethan Brown,ethan.brown@example.com,Coffee Maker,Home,3,40.70,2024-01-01,Seattle,Debit Card,122.10
1,1118,Priya Brown,priya.brown@example.com,Notebook Set,Stationery,1,8.19,2024-01-03,New York,Credit Card,8.19
2,1080,Oliver Khan,oliver.khan@example.com,Coffee Maker,Home,1,39.25,2024-01-05,New York,Cash,39.25
3,1064,Anaya Smith,anaya.smith@example.com,Bluetooth Speaker,Electronics,4,43.64,2024-01-09,Miami,Cash,174.56
4,1115,Oliver Johnson,oliver.johnson@example.com,Backpack,Accessories,1,55.04,2024-01-09,Unknown,Debit Card,55.04
5,1197,Lucas Johnson,lucas.johnson@example.com,Running Shoes,Fitness,2,62.42,2024-01-10,Los Angeles,Cash,124.84
6,1147,Priya Miller,priya.miller@example.com,Desk Lamp,Home,4,18.27,2024-01-11,Los Angeles,Credit Card,73.08
7,1063,Ethan Johnson,ethan.johnson@example.com,Sunglasses,Accessories,1,29.35,2024-01-13,Seattle,Cash,29.35
8,1210,Priya Chen,priya.chen@example.com,Notebook Set,Stationery,4,8.40,2024-01-13,Miami,Credit Card,33.60
9,1214,Anaya Patel,Unknown,Yoga Mat,Fitness,4,21.56,2024-01-14,Houston,Debit Card,86.24


In [196]:
print(f"Final cleaned shape: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")
df_clean.describe(include="all").T

Final cleaned shape: 220 rows x 11 columns


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
OrderID,220.0,NaN,NaN,NaN,1110.5,1001.0,1055.75,1110.5,1165.25,1220.0,63.652704
CustomerName,220,140,Unknown,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Email,220,141,Unknown,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product,220,10,Coffee Maker,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,220,5,Home,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,220.0,NaN,NaN,NaN,2.95,1.0,2.0,3.0,4.0,5.0,1.408471
Price,220.0,NaN,NaN,NaN,31.830455,6.0,19.1625,29.53,44.155,67.66,17.540091
OrderDate,220,NaN,NaN,NaN,2024-06-11 11:27:16.363636,2024-01-01 00:00:00,2024-03-29 00:00:00,2024-06-13 00:00:00,2024-09-03 12:00:00,2024-10-26 00:00:00,NaN
ShippingCity,220,7,Miami,41,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PaymentMethod,220,5,Credit Card,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Before vs. after summary

In [197]:
summary = pd.DataFrame({
    "Raw dataset": [df.shape[0], df.isnull().sum().sum(), df.duplicated().sum()],
    "Cleaned dataset": [df_clean.shape[0], df_clean.isnull().sum().sum(), df_clean.duplicated().sum()],
}, index=["Rows", "Missing values", "Duplicate rows"])
summary

,Raw dataset,Cleaned dataset
Rows,232,220
Missing values,53,0
Duplicate rows,12,0


## Bonus — Save the cleaned dataset as a new CSV file

In [198]:
df_clean.to_csv("data/online_retail_orders_cleaned.csv", index=False)
print("Saved cleaned dataset to data/online_retail_orders_cleaned.csv")

Saved cleaned dataset to data/online_retail_orders_cleaned.csv


## Conclusion

Starting from a 232-row raw export, the cleaning pipeline:

- **Standardized** inconsistent text casing/whitespace in `CustomerName`,
  `ShippingCity`, `PaymentMethod`, and `Category`.
- **Converted** `Price` (mixed `$` / text / whitespace) and `Quantity`
  (mixed text like `"3 units"`, and negative values) into proper numeric
  types.
- **Unified** three different `OrderDate` formats into a single `datetime64` column.
- **Imputed** missing values (median for numeric columns, an explicit
  `"Unknown"` flag for categorical/identifier columns rather than guessing).
- **Removed** duplicate rows.
- **Added** a derived `TotalAmount` column and produced an analysis-ready,
  correctly-typed DataFrame, saved to `data/online_retail_orders_cleaned.csv`.
